# Custom Multi-Asset Index Construction & Rebalancing with `yfinance`

This notebook builds a **custom momentum-based multi-asset index** using ETF proxies and compares it against the **S&P 500 ETF (`SPY`)**.

## What this notebook includes
- Data download with `yfinance`
- A clear **index methodology**
- **Monthly rebalancing**
- **Momentum-based asset selection**
- Portfolio/index return construction
- Performance comparison vs `SPY`
- Sharpe ratio and other useful metrics
- Clean visualizations

## Strategy idea
We will:
1. Build a **multi-asset universe** using liquid ETFs.
2. Use **trailing 6-month momentum** as the selection signal.
3. Rebalance **monthly**.
4. Select the **top 3 assets** by momentum.
5. Weight selected assets **equally** until the next rebalance.

This is a simple and transparent index methodology that is easy to explain, audit, and extend.

In [1]:
# If needed, uncomment and run:
# !pip install yfinance pandas numpy matplotlib

import warnings
warnings.filterwarnings("ignore")

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True

ModuleNotFoundError: No module named 'matplotlib'

## 1. Define the universe and backtest settings

We use ETFs as **asset-class proxies**:

- `SPY`: U.S. equities
- `TLT`: Long-term U.S. Treasuries
- `GLD`: Gold
- `VNQ`: U.S. REITs
- `DBC`: Broad commodities

You can easily replace or expand this universe later.

In [2]:
# -----------------------------
# Backtest configuration
# -----------------------------
UNIVERSE = ["SPY", "TLT", "GLD", "VNQ", "DBC"]
BENCHMARK = "SPY"

START_DATE = "2012-01-01"
END_DATE = None            # None = up to latest available date

MOMENTUM_LOOKBACK = 126    # ~6 trading months
TOP_N = 3                  # number of assets selected each rebalance
REBALANCE_FREQ = "M"       # month-end rebalance
RISK_FREE_RATE = 0.0       # annualized, for Sharpe ratio

## 2. Download price data

We use **adjusted close prices** because they account for splits and dividends more appropriately for backtesting.

In [ ]:
# Download adjusted prices
raw = yf.download(
    tickers=UNIVERSE,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)

# yfinance sometimes returns a multi-index column structure
if isinstance(raw.columns, pd.MultiIndex):
    if "Close" in raw.columns.get_level_values(0):
        prices = raw["Close"].copy()
    else:
        prices = raw.xs(raw.columns.levels[0][0], axis=1, level=0).copy()
else:
    prices = raw.copy()

# Keep only the requested tickers and drop rows with all NaNs
prices = prices[UNIVERSE].dropna(how="all")

# Forward-fill occasional missing values
prices = prices.ffill().dropna()

prices.tail()

Ticker,SPY,TLT,GLD,VNQ,DBC
Date,,,,,
2026-05-01,720.650024,85.610001,423.179993,96.059998,30.809999
2026-05-04,718.010010,84.959999,414.709991,95.459999,31.330000
2026-05-05,723.770020,85.430000,418.269989,95.760002,31.200001
2026-05-06,733.830017,86.080002,430.959991,97.089996,30.209999
2026-05-07,731.580017,85.650002,431.679993,96.389999,30.250000


## 3. Compute daily returns

The index will ultimately be built from daily asset returns, but weights will only change at monthly rebalance dates.

In [4]:
daily_returns = prices.pct_change().fillna(0.0)
daily_returns.head()

Ticker,SPY,TLT,GLD,VNQ,DBC
Date,,,,,
2012-01-03,0.000000,0.000000,0.000000,0.000000,0.000000
2012-01-04,0.001569,-0.011890,0.005067,-0.017097,0.007598
2012-01-05,0.002662,-0.001780,0.006828,0.009393,-0.013645
2012-01-06,-0.002577,0.007895,-0.003676,-0.003274,0.004368
2012-01-09,0.002428,-0.001769,-0.004453,-0.003458,0.001087


## 4. Define the momentum signal

We use **trailing 6-month return**:

\[
\text{Momentum}_{t} = \frac{P_t}{P_{t-L}} - 1
\]

where:
- \( P_t \) is the latest price available at the rebalance date
- \( L \) is the lookback window (`126` trading days here)

To avoid look-ahead bias, the weights chosen on a rebalance date are based only on information available **up to that date**.

In [5]:
momentum = prices.pct_change(MOMENTUM_LOOKBACK)
momentum.tail()

Ticker,SPY,TLT,GLD,VNQ,DBC
Date,,,,,
2026-05-01,0.054355,-0.035533,0.165785,0.106661,0.402331
2026-05-04,0.062175,-0.037255,0.120444,0.094556,0.425372
2026-05-05,0.067195,-0.029034,0.136233,0.095164,0.409531
2026-05-06,0.080002,-0.019193,0.168610,0.111996,0.354740
2026-05-07,0.089606,-0.026263,0.191433,0.102122,0.366016


## 5. Create monthly rebalance dates

We rebalance at the **last trading day of each month**.

In [6]:
# Last available trading day in each calendar month
rebalance_dates = prices.resample(REBALANCE_FREQ).last().index

# Keep only dates that exist in the trading calendar
rebalance_dates = [d for d in rebalance_dates if d in prices.index]

rebalance_dates[:5], rebalance_dates[-5:]

([Timestamp('2012-01-31 00:00:00'),
  Timestamp('2012-02-29 00:00:00'),
  Timestamp('2012-04-30 00:00:00'),
  Timestamp('2012-05-31 00:00:00'),
  Timestamp('2012-07-31 00:00:00')],
 [Timestamp('2025-09-30 00:00:00'),
  Timestamp('2025-10-31 00:00:00'),
  Timestamp('2025-12-31 00:00:00'),
  Timestamp('2026-03-31 00:00:00'),
  Timestamp('2026-04-30 00:00:00')])

## 6. Build the index methodology

### Methodology
At each monthly rebalance:
1. Read the momentum score for each asset.
2. Rank assets from highest to lowest momentum.
3. Select the **top `TOP_N`** assets.
4. Assign **equal weights** to selected assets.
5. Hold those weights until the next rebalance.

If an asset has missing momentum on a rebalance date, it is excluded from selection for that date.

In [ ]:
def compute_rebalance_weights(momentum_df, rebalance_dates, top_n=3):
    '''
    Return a DataFrame of target weights on rebalance dates.
    '''
    weights = pd.DataFrame(
        0.0,
        index=pd.DatetimeIndex(rebalance_dates),
        columns=momentum_df.columns
    )

    for dt in weights.index:
        signal = momentum_df.loc[dt].dropna()
        if signal.empty:
            continue

        selected = signal.sort_values(ascending=False).head(top_n).index
        weights.loc[dt, selected] = 1.0 / len(selected)

    return weights

target_weights = compute_rebalance_weights(momentum, rebalance_dates, top_n=TOP_N)
target_weights.tail()

Ticker,SPY,TLT,GLD,VNQ,DBC
2025-09-30,0.333333,0.0,0.333333,0.333333,0.000000
2025-10-31,0.333333,0.0,0.333333,0.000000,0.333333
2025-12-31,0.333333,0.0,0.333333,0.000000,0.333333
2026-03-31,0.000000,0.0,0.333333,0.333333,0.333333
2026-04-30,0.000000,0.0,0.333333,0.333333,0.333333


## 7. Expand target weights to daily holdings

The target weights are only defined on rebalance dates.  
To compute daily portfolio returns, we forward-fill weights between rebalance dates.

In [8]:
# Reindex weights to all trading dates and forward-fill between rebalances
daily_weights = target_weights.reindex(prices.index).ffill().fillna(0.0)

# Shift by 1 day so that returns on day t are earned using weights decided at the close of day t-1
daily_weights = daily_weights.shift(1).fillna(0.0)

daily_weights.head()

Ticker,SPY,TLT,GLD,VNQ,DBC
Date,,,,,
2012-01-03,0.0,0.0,0.0,0.0,0.0
2012-01-04,0.0,0.0,0.0,0.0,0.0
2012-01-05,0.0,0.0,0.0,0.0,0.0
2012-01-06,0.0,0.0,0.0,0.0,0.0
2012-01-09,0.0,0.0,0.0,0.0,0.0


## 8. Compute index returns

The custom index daily return is:

\[
r^{index}_t = \sum_i w_{i,t} \cdot r_{i,t}
\]

where:
- \( w_{i,t} \) is the weight in asset \( i \) on day \( t \)
- \( r_{i,t} \) is the asset's daily return

In [9]:
strategy_returns = (daily_weights * daily_returns).sum(axis=1)
strategy_returns.name = "Custom Multi-Asset Index"

benchmark_returns = daily_returns[BENCHMARK].copy()
benchmark_returns.name = BENCHMARK

returns_df = pd.concat([strategy_returns, benchmark_returns], axis=1).dropna()
returns_df.head()

,Custom Multi-Asset Index,SPY
Date,,
2012-01-03,0.0,0.000000
2012-01-04,0.0,0.001569
2012-01-05,0.0,0.002662
2012-01-06,0.0,-0.002577
2012-01-09,0.0,0.002428


## 9. Convert returns into index levels

To make the series easier to compare visually, we convert each return stream into a normalized index level starting at **100**.

In [10]:
def returns_to_index(returns, start_value=100.0):
    return start_value * (1 + returns).cumprod()

index_levels = pd.concat(
    [
        returns_to_index(returns_df["Custom Multi-Asset Index"]).rename("Custom Multi-Asset Index"),
        returns_to_index(returns_df[BENCHMARK]).rename(BENCHMARK)
    ],
    axis=1
)

index_levels.head()

# =========================================
# 9b. Blended Benchmark
# =========================================

# Average over rebalance dates only (monthly_weights from section 5)
# Reconstruct from valid_dates (defined in section 4, always in scope)
avg_weights = weights.loc[valid_dates].mean()
avg_weights = avg_weights / avg_weights.sum()

print("\n===== Average Allocation across all rebalances =====")
for ticker, wt in avg_weights.items():
    print(f"  {ticker}: {wt:.1%}")

# Apply static avg weights to daily returns
avg_bench_returns = (returns[avg_weights.index] * avg_weights).sum(axis=1)
avg_bench_returns = avg_bench_returns[portfolio_returns.index]
avg_bench_cum = (1 + avg_bench_returns).cumprod()

avg_bench_return = avg_bench_cum.iloc[-1] - 1
avg_bench_sharpe = sharpe_ratio(avg_bench_returns, risk_free_rate)
avg_bench_mdd    = max_drawdown(avg_bench_cum)

vs_avg_return = ((index_return - avg_bench_return) / abs(avg_bench_return)) * 100
vs_avg_sharpe = ((index_sharpe - avg_bench_sharpe) / abs(avg_bench_sharpe)) * 100

print("\n===== Blended Benchmark Comparison =====")
print(f"{'Metric':<25} {'Custom Index':>14} {'Avg-Wt Bench':>14} {'SPY':>10}")
print("-" * 65)
print(f"  {'Total Return':<23} {index_return:>14.2%} {avg_bench_return:>14.2%} {spy_return:>10.2%}")
print(f"  {'Sharpe Ratio':<23} {index_sharpe:>14.2f} {avg_bench_sharpe:>14.2f} {spy_sharpe:>10.2f}")
print(f"  {'Max Drawdown':<23} {index_mdd:>14.2%} {avg_bench_mdd:>14.2%} {spy_mdd:>10.2%}")
print(f"\n  Index vs Avg-Weight Benchmark:")
print(f"    Return outperformance : {vs_avg_return:+.2f}%")
print(f"    Sharpe improvement    : {vs_avg_sharpe:+.2f}%")

NameError: name 'weights' is not defined

## 10. Performance metrics

We compute:
- **Total Return**
- **CAGR**
- **Annualized Volatility**
- **Sharpe Ratio**
- **Maximum Drawdown**

### Sharpe Ratio
We use the standard approximation:

\[
\text{Sharpe} = \frac{\mu_p - r_f}{\sigma_p}
\]

where returns are annualized from daily data.

In [12]:
def max_drawdown(index_series):
    running_max = index_series.cummax()
    drawdown = index_series / running_max - 1.0
    return drawdown.min()

def performance_summary(returns, rf=0.0, trading_days=252):
    returns = returns.dropna()
    index_series = returns_to_index(returns)

    total_return = index_series.iloc[-1] / index_series.iloc[0] - 1.0
    n_years = len(returns) / trading_days
    cagr = (index_series.iloc[-1] / index_series.iloc[0]) ** (1 / n_years) - 1.0 if n_years > 0 else np.nan
    ann_vol = returns.std() * np.sqrt(trading_days)
    ann_return = returns.mean() * trading_days
    sharpe = (ann_return - rf) / ann_vol if ann_vol != 0 else np.nan
    mdd = max_drawdown(index_series)

    return pd.Series({
        "Total Return": total_return,
        "CAGR": cagr,
        "Annualized Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": mdd
    })

summary = pd.concat(
    [
        performance_summary(returns_df["Custom Multi-Asset Index"], rf=RISK_FREE_RATE).rename("Custom Multi-Asset Index"),
        performance_summary(returns_df[BENCHMARK], rf=RISK_FREE_RATE).rename(BENCHMARK)
    ],
    axis=1
).T

summary

# =========================================
# 10. Plots
# =========================================

# --- Plot 1: Cumulative returns (now includes blended benchmark) ---
plt.figure(figsize=(12, 6))
plt.plot(portfolio_cum,  label="Custom Dual Momentum Index", linewidth=2)
plt.plot(avg_bench_cum,  label="Avg-Weight Benchmark (fair baseline)", linewidth=2, linestyle="--")
plt.plot(spy_cum,        label="SPY",                                  linewidth=2, linestyle=":")
plt.title("Custom Multi-Asset Index vs Benchmarks")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# --- Plot 2: Rolling Sharpe (unchanged) ---
rolling_sharpe_index = portfolio_returns.rolling(252).apply(
    lambda x: sharpe_ratio(pd.Series(x), risk_free_rate), raw=False
)
rolling_sharpe_spy = spy_returns.rolling(252).apply(
    lambda x: sharpe_ratio(pd.Series(x), risk_free_rate), raw=False
)
rolling_sharpe_avg = avg_bench_returns.rolling(252).apply(
    lambda x: sharpe_ratio(pd.Series(x), risk_free_rate), raw=False
)

plt.figure(figsize=(12, 6))
plt.plot(rolling_sharpe_index, label="Custom Index Rolling Sharpe", linewidth=2)
plt.plot(rolling_sharpe_avg,   label="Avg-Weight Benchmark Sharpe", linewidth=2, linestyle="--")
plt.plot(rolling_sharpe_spy,   label="SPY Rolling Sharpe",          linewidth=2, linestyle=":")
plt.title("Rolling 1Y Sharpe Ratio")
plt.xlabel("Date")
plt.ylabel("Sharpe Ratio")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# --- Plot 3: Drawdown (unchanged) ---
drawdown_index = portfolio_cum / portfolio_cum.cummax() - 1
drawdown_spy   = spy_cum   / spy_cum.cummax()   - 1
drawdown_avg   = avg_bench_cum / avg_bench_cum.cummax() - 1

plt.figure(figsize=(12, 6))
plt.plot(drawdown_index, label="Custom Index Drawdown",        linewidth=2)
plt.plot(drawdown_avg,   label="Avg-Weight Benchmark Drawdown",linewidth=2, linestyle="--")
plt.plot(drawdown_spy,   label="SPY Drawdown",                 linewidth=2, linestyle=":")
plt.title("Drawdown Comparison")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

,Total Return,CAGR,Annualized Volatility,Sharpe Ratio,Max Drawdown
Custom Multi-Asset Index,3.412849,0.109316,0.108575,1.010059,-0.214030
SPY,6.389122,0.150007,0.166081,0.925021,-0.337173


## 11. Rebalance history

This table shows which assets were held at each rebalance date and in what weights.

In [13]:
rebalance_history = target_weights[target_weights.sum(axis=1) > 0].copy()
rebalance_history.tail(12)

Ticker,SPY,TLT,GLD,VNQ,DBC
2024-12-31,0.333333,0.000000,0.333333,0.333333,0.000000
2025-01-31,0.333333,0.000000,0.333333,0.000000,0.333333
2025-02-28,0.333333,0.000000,0.333333,0.000000,0.333333
2025-03-31,0.333333,0.000000,0.333333,0.000000,0.333333
2025-04-30,0.333333,0.333333,0.333333,0.000000,0.000000
2025-06-30,0.333333,0.000000,0.333333,0.000000,0.333333
2025-07-31,0.333333,0.000000,0.333333,0.000000,0.333333
2025-09-30,0.333333,0.000000,0.333333,0.333333,0.000000
2025-10-31,0.333333,0.000000,0.333333,0.000000,0.333333
2025-12-31,0.333333,0.000000,0.333333,0.000000,0.333333


## 12. Visualization: index performance vs SPY

In [ ]:
ax = index_levels.plot(title="Custom Momentum Index vs SPY", linewidth=2)
ax.set_ylabel("Index Level (Start = 100)")
plt.show()

## 13. Visualization: drawdowns

In [14]:
def compute_drawdown_series(index_series):
    running_max = index_series.cummax()
    return index_series / running_max - 1.0

drawdowns = index_levels.apply(compute_drawdown_series)

ax = drawdowns.plot(title="Drawdowns", linewidth=2)
ax.set_ylabel("Drawdown")
plt.show()

TypeError: expected string or bytes-like object, got 'NoneType'

## 14. Visualization: weights through time

In [ ]:
ax = daily_weights.plot.area(title="Daily Portfolio Weights", alpha=0.8)
ax.set_ylabel("Weight")
plt.show()

## 15. Visualization: rolling Sharpe ratio

This helps evaluate whether the strategy's risk-adjusted performance was stable over time.

In [ ]:
ROLLING_WINDOW = 252  # 1 year of trading days

rolling_sharpe = (
    (returns_df.rolling(ROLLING_WINDOW).mean() * 252 - RISK_FREE_RATE) /
    (returns_df.rolling(ROLLING_WINDOW).std() * np.sqrt(252))
)

ax = rolling_sharpe.plot(title="Rolling 1-Year Sharpe Ratio", linewidth=2)
ax.set_ylabel("Sharpe Ratio")
plt.show()

## 16. Interpretation

This project demonstrates a complete workflow for a **custom multi-asset index construction** problem:

- Use market data from `yfinance`
- Define a transparent **index methodology**
- Select assets using a **momentum factor**
- Rebalance **monthly**
- Measure performance against a benchmark (`SPY`)
- Evaluate both return and risk-adjusted metrics such as **Sharpe ratio**

## Possible extensions
You can improve or customize this framework by adding:
- Transaction costs
- Turnover analysis
- Volatility scaling
- Risk-parity weighting instead of equal weighting
- Absolute momentum filters (for example, move to cash if momentum is negative)
- More asset classes (international equity, short duration bonds, FX, crypto proxies, etc.)
- Walk-forward parameter testing

## 17. Optional: export results

In [ ]:
# Save outputs if desired
summary.to_csv("performance_summary.csv")
rebalance_history.to_csv("rebalance_history.csv")
index_levels.to_csv("index_levels.csv")

print("Saved: performance_summary.csv, rebalance_history.csv, index_levels.csv")

## 18. Final notes

This notebook is intentionally designed to be:
- Easy to explain in an interview or project presentation
- Easy to modify for more sophisticated index methodologies
- Clean enough to serve as a portfolio project

A good next step would be to package the main logic into reusable functions or a class so the strategy can be tested on multiple universes and parameter settings.